# 02_clustering.py
# Edu-Platform — K-Means Clustering Pipeline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pickle
import os
import warnings

warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

plt.style.use('seaborn-v0_8-whitegrid')
os.makedirs('../models', exist_ok=True)
os.makedirs('../plots', exist_ok=True)

## 📌 Step 1: Data Load

In [ ]:
df = pd.read_csv('../../data/sample_aptitude_data.csv')

features = ['logical', 'verbal', 'numerical', 'memory', 'attention']
X = df[features].copy()

print(X.head())

## 📌 Step 2: Feature Scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=features)

with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

## 📌 Step 3: Optimal K

In [ ]:
K_range = range(2, 11)
wcss = []
silhouette_scores = []
calinski_scores = []
davies_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)

    wcss.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))
    calinski_scores.append(calinski_harabasz_score(X_scaled, labels))
    davies_scores.append(davies_bouldin_score(X_scaled, labels))

In [ ]:
plt.plot(K_range, wcss)
plt.title("Elbow Method")
plt.show()

## 📌 Step 4: Train Model

In [ ]:
OPTIMAL_K = 4

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)

df['cluster_id'] = cluster_labels

In [ ]:
centers_df = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=features
)

cluster_feature_max = centers_df.idxmax(axis=1)

feature_to_style = {
    'verbal': 'visual_learner',
    'logical': 'conceptual_thinker',
    'numerical': 'practice_based',
    'memory': 'step_by_step',
    'attention': 'practice_based'
}

CLUSTER_MAP = {}
for cid in range(OPTIMAL_K):
    CLUSTER_MAP[cid] = feature_to_style.get(cluster_feature_max[cid])

df['learning_style'] = df['cluster_id'].map(CLUSTER_MAP)

## 📌 Step 5: PCA Visualization

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels)
plt.show()

## 📌 Step 6: Analysis

In [ ]:
for cid in range(OPTIMAL_K):
    cluster_data = df[df['cluster_id'] == cid]
    print(cluster_data[features].mean())

## 📌 Step 7: Save Model

In [ ]:
with open('../models/kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

with open('../models/cluster_map.pkl', 'wb') as f:
    pickle.dump(CLUSTER_MAP, f)

with open('../models/pca_model.pkl', 'wb') as f:
    pickle.dump(pca, f)

df.to_csv('../../data/clustered_students.csv', index=False)

In [ ]:
new_student = {
    'logical': 78,
    'verbal': 45,
    'numerical': 88,
    'memory': 60,
    'attention': 85
}

student_features = [[new_student[f] for f in features]]
student_scaled = scaler.transform(student_features)

predicted_cluster = kmeans.predict(student_scaled)[0]
predicted_style = CLUSTER_MAP[predicted_cluster]

print(predicted_cluster, predicted_style)